# Create delta tables of European Countries

## Load foundation

In [0]:
%run "./foundation"


## Loading raw data from source

In [0]:
df_countries = spark.read.format('csv') \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load("abfss://source@stiidigiiar01dev.dfs.core.windows.net/countries/countries.csv")

In [0]:
df_countries.display()

### PySpark Transformation

In [0]:
df_europe = df_countries.filter(df_countries.region == "Europe")

In [0]:
df_europe.display()

## Create tables based on the preprocessed data

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS experimentDB

In [0]:
%sql
CREATE TABLE IF NOT EXISTS experimentDB.europe_countries
(
    `name` STRING,
    `alpha-2` CHAR(2),
    `alpha-3` CHAR(3),
    `country-code` CHAR(3),
    `iso_3166-2` STRING,
    `region` STRING,
    `sub-region` STRING
)
USING DELTA
LOCATION 'abfss://lvl-2@stiidigiiar01dev.dfs.core.windows.net/experimentDB/europe_countries'

In [0]:
from pyspark.sql.functions import col

df_europe.select("name", "alpha-2", "alpha-3", col("`country-code`").cast("string").alias("country-code"), "iso_3166-2", "region", "sub-region") \
    .write.format("delta") \
    .mode("append") \
    .saveAsTable(f"experimentDB.europe_countries")

In [0]:
%sql
SELECT * FROM experimentDB.europe_countries

In [0]:
%sql
DROP TABLE experimentDB.europe_countries